In [ ]:
import os
from openai import AzureOpenAI
from tqdm import tqdm
import time
import re
from sklearn.metrics import *
from krippendorff import alpha as k_alpha

# Define Endpoint and Query Function

In [ ]:
api_key = os.getenv("AZURE_OPENAI_API_KEY_MODEL")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_MODEL")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION_MODEL")

In [ ]:
# client = AzureOpenAI(
#     azure_endpoint=endpoint,
#     api_key=api_key,
#     api_version=api_version,
# )

In [ ]:
# response = client.chat.completions.create(
#     model=deployment_name,
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "What is the capital of France?"}
#     ]
# )

In [ ]:
# print(response.choices[0].message.content)

In [ ]:
import os
from openai import AzureOpenAI

endpoint = ""
model_name = "gpt-5.2-chat"
deployment_name = "gpt-5.2-chat"

api_key = ""
api_version = ""

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_completion_tokens =16384,
    model=deployment_name,
    reasoning_effort="none",
)

print(response.choices[0].message.content)

In [ ]:
def query_llm(sys_msg, query_msg, dt=2e-2):
    time.sleep(dt)
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": query_msg}
            ],
            # reasoning_effort="none",
        )
        return response.choices[0].message.content
    except Exception as e:
        print(e)
        return -1

# Load LLM Response Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
file_path = "emotions_train_df.csv"

In [ ]:
data = pd.read_csv(file_path)
data

# Query for LLM

In [ ]:
class_cols = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral']
class_cols = np.array(class_cols)

In [ ]:
system_prompt = """You will be given a text, and your task is to classify it into a sentiment.
This is the list of sentiments you can choose from:
admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral.

Just output one of the emotions. There is no need to provide any explanation.
"""

query = """<Start of text>
{}
<End of text>

The type of sentiment the text display is: """

In [ ]:
def get_llm_prompt(text):
    user_query = query.format(text)
    return system_prompt, user_query

In [ ]:
sys_prompt, query_prompt = get_llm_prompt(data['text'][0])

In [ ]:
query_prompt

In [ ]:
response = query_llm(sys_msg=sys_prompt, query_msg=query_prompt)
response

In [ ]:
data['label'][0]

# DSPY Attempt

In [ ]:
import dspy
from dspy.teleprompt import *
import random
from sklearn.model_selection import train_test_split

In [ ]:
lm = dspy.LM(f"{deployment_name}", api_key=api_key, base_url=endpoint, api_version=api_version, temperature=1, max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
lm("Say this is a test!")  # => ['This is a test!']

In [ ]:
program = dspy.Predict("instruction: str, prompt: str -> emotion_type: str")
program

In [ ]:
i = 15
sys_msg, q_msg = get_llm_prompt(data['text'][i])
program(instruction=sys_msg, prompt=q_msg)

In [ ]:
data['label'][i]

In [ ]:
def get_index_from_answer(answer):
    return (data['text'] == answer).idxmax()

In [ ]:
len("<Start of text>\n")

In [ ]:
def compute_lm_accuracy(example, prediction, trace=None):
    try:
        pred_label = prediction.scores
        
        # Extract the index based on the prompt
        prompt = example.prompt
        prompt = prompt.split("<End of text>")[0]
        prompt = prompt[16:]
        idx = get_index_from_answer(prompt)
        target = data['label'].iloc[idx]

        return target == pred_label
    except Exception as e:
        return -1

In [ ]:
dataset = []
for i in range(data.shape[0]):
    if i % 10 != 0:
        continue
    sys_msg, q_msg = get_llm_prompt(data['text'][i])
    dataset.append(dspy.Example(instruction=sys_msg, prompt=q_msg).with_inputs("instruction", "prompt"))

trainset, valset = train_test_split(dataset, test_size=0.2, random_state=3407)
len(trainset), len(valset)

In [ ]:
opt = MIPROv2(
    metric=compute_lm_accuracy,
    # max_steps=16,
    # max_demos=8,
    auto="heavy",
    max_bootstrapped_demos=16,
    max_labeled_demos=8,
)

# compile/optimize the program (this only searches prompt/instruction space)
optimized_program = opt.compile(
    program, 
    trainset=trainset,
    valset=valset
)

optimized_program.save(f"./emotions_dspy_program_gpt-5point2-chat/", save_program=True)

In [ ]:
dspy.inspect_history()

In [ ]:
optimized_program = dspy.load("./emotions_dspy_program_gpt-5point2-chat/", allow_pickle=True)

In [ ]:
# Example usage: call the optimized program. We must use your get_llm_input to create the prompts.
# The optimized program will still call your configured `lm` under the hood.
i = 63
system_prompt, query_prompt = get_llm_prompt(data['text'][i])
out = optimized_program(instruction=system_prompt, prompt=query_prompt)
print("Scores:", out.emotion_type)

In [ ]:
data['label'].iloc[i]

In [ ]:
test_data = pd.read_csv("emotions_test_df.csv")

In [ ]:
import time
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from collections import Counter

def predict_threaded(test_data,
                             get_llm_prompt,
                             optimized_program,
                             max_workers=8,
                             max_retries=2,
                             retry_backoff=2.0,
                             default_on_error=-1,
                             non_retriable_substrings=None,
                             show_summary=True):
    """
    Threaded predictions preserving order. Always returns length == test_data.shape[0].
    Failures are replaced by `default_on_error`.
    """
    if non_retriable_substrings is None:
        non_retriable_substrings = [
            "contentpolicy", "contentpolicyviolation", "contentpolicyviolationerror",
            "filtered due to the prompt", "content management policy", "policy violation"
        ]

    def is_non_retriable(exc_text):
        txt = exc_text.lower()
        return any(sub in txt for sub in non_retriable_substrings)

    n = int(test_data.shape[0])
    results = [None] * n

    def worker(i):
        text = test_data['text'][i]
        system_prompt, query_prompt = get_llm_prompt(text)
        attempt = 0
        while True:
            try:
                return optimized_program(instruction=system_prompt, prompt=query_prompt)
            except Exception as e:
                s = str(e)
                # treat content-policy-like errors as non-retriable -> immediate sentinel
                if is_non_retriable(s):
                    return default_on_error
                attempt += 1
                if attempt > max_retries:
                    return default_on_error
                time.sleep(retry_backoff ** attempt)

    # submit
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_to_index = {ex.submit(worker, i): i for i in range(n)}
        completed_count = 0
        failed_count = 0
        with tqdm(total=n) as pbar:
            for fut in as_completed(future_to_index):
                i = future_to_index[fut]
                try:
                    out = fut.result()
                except Exception as e:
                    # Unexpected failure (should be rare because worker returns sentinel on failure)
                    results[i] = default_on_error
                    failed_count += 1
                else:
                    # store output (could be sentinel returned by worker)
                    results[i] = out
                    if out == default_on_error:
                        failed_count += 1
                completed_count += 1
                pbar.update(1)

    if show_summary:
        print(f"submitted: {n}, completed: {completed_count}, failed (== sentinel): {failed_count}")
        # optional quick histogram of output lengths (chars)
        out_lens = [len(str(x)) if x is not None else 0 for x in results]
        buckets = [0,50,100,200,500,1000]
        def bucket_counts(vals, bins):
            from collections import Counter
            cnt = Counter()
            for v in vals:
                for b in bins:
                    if v <= b:
                        cnt[b] += 1
                        break
                else:
                    cnt["+"+str(bins[-1])] += 1
            return dict(cnt)
        print("output length buckets (chars):", bucket_counts(out_lens, buckets))

    return results

In [ ]:
pred = predict_threaded(test_data, get_llm_prompt, optimized_program, max_workers=24)
len(pred)

In [ ]:
mapped_pred = []
for p in pred:
    try:
        mapped_pred.append(p.emotion_type)
    except Exception as e:
        mapped_pred.append("neutral")

In [ ]:
(mapped_pred == test_data['label']).mean()

In [ ]:
balanced_accuracy_score(test_data['label'], mapped_pred)

In [ ]:
f1_score(test_data['label'], mapped_pred, average='macro')

In [ ]:
print(classification_report(test_data['label'], mapped_pred))

In [ ]:
# optimized_program.save(f"./emotions_dspy_program_gpt-4o/", save_program=True)